# How does the weather affect the power generation?

This notebook focuses on quantifying the relationship between weather variables (Irradiation, Temperature) and power output (DC/AC Power).

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set plot style
plt.style.use('ggplot')
sns.set_palette('magma')

ModuleNotFoundError: No module named 'scipy'

## 1. Data Preparation

We merge generation and weather data for both plants to create a comprehensive dataset for analysis.

In [ ]:
# Paths
p1_gen_path = '../../data/Plant_1_Generation_Data (2).csv'
p2_gen_path = '../../data/Plant_2_Generation_Data (2).csv'
p1_weather_path = '../../data/Plant_1_Weather_Sensor_Data (2).csv'
p2_weather_path = '../../data/Plant_2_Weather_Sensor_Data (1).csv'

# Load and Preprocess
def load_and_merge(gen_path, weather_path, plant_name, dayfirst=False):
    gen = pd.read_csv(gen_path)
    weather = pd.read_csv(weather_path)
    
    gen['DATE_TIME'] = pd.to_datetime(gen['DATE_TIME'], dayfirst=dayfirst)
    weather['DATE_TIME'] = pd.to_datetime(weather['DATE_TIME'])
    
    # Aggregate generation to match weather sensor timestamps
    gen_agg = gen.groupby('DATE_TIME')[['DC_POWER', 'AC_POWER', 'DAILY_YIELD']].mean().reset_index()
    
    merged = pd.merge(gen_agg, weather, on='DATE_TIME')
    merged['PLANT'] = plant_name
    return merged

df1 = load_and_merge(p1_gen_path, p1_weather_path, 'Plant_1', dayfirst=True)
df2 = load_and_merge(p2_gen_path, p2_weather_path, 'Plant_2', dayfirst=False)

df = pd.concat([df1, df2])
df.head()

## 2. Irradiation: The Primary Driver

Irradiation is the most direct cause of power generation. Let's quantify this relationship.

In [ ]:
plt.figure(figsize=(12, 6))
sns.lmplot(data=df, x='IRRADIATION', y='DC_POWER', hue='PLANT', scatter_kws={'alpha':0.3}, height=6, aspect=1.5)
plt.title('Impact of Irradiation on DC Power Generation')
plt.show()

for plant in ['Plant_1', 'Plant_2']:
    plant_data = df[df['PLANT'] == plant]
    correlation = plant_data['IRRADIATION'].corr(plant_data['DC_POWER'])
    print(f"{plant} Irradiation-DC Correlation: {correlation:.4f}")

## 3. Temperature and Efficiency

Solar panels become less efficient as they heat up. We can analyze this by looking at the ratio of `DC_POWER` to `IRRADIATION` (a proxy for efficiency) against `MODULE_TEMPERATURE`.

In [ ]:
# Filter out rows with zero irradiation to avoid division by zero
efficiency_df = df[df['IRRADIATION'] > 0.01].copy()
efficiency_df['EFFICIENCY_PROXY'] = efficiency_df['DC_POWER'] / efficiency_df['IRRADIATION']

plt.figure(figsize=(12, 6))
sns.scatterplot(data=efficiency_df, x='MODULE_TEMPERATURE', y='EFFICIENCY_PROXY', hue='PLANT', alpha=0.4)
plt.title('Module Temperature vs. Generation Efficiency Proxy')
plt.ylabel('DC Power / Irradiation')
plt.show()

print("Note: In Plant 1, the DC power scale is much higher, which is why the efficiency proxy values differ from Plant 2.")

## 4. Case Study: Sunny vs. Cloudy Day

How does generation look on a high-irradiation day versus a low-irradiation day?

In [ ]:
df['DATE'] = df['DATE_TIME'].dt.date
daily_irr = df.groupby(['DATE', 'PLANT'])['IRRADIATION'].sum().reset_index()

sunny_day = daily_irr.loc[daily_irr['IRRADIATION'].idxmax()]['DATE']
cloudy_day = daily_irr.loc[daily_irr['IRRADIATION'].idxmin()]['DATE']

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

sns.lineplot(data=df[df['DATE'] == sunny_day], x='DATE_TIME', y='DC_POWER', hue='PLANT', ax=axes[0])
axes[0].set_title(f'Power Generation on a Sunny Day ({sunny_day})')

sns.lineplot(data=df[df['DATE'] == cloudy_day], x='DATE_TIME', y='DC_POWER', hue='PLANT', ax=axes[1])
axes[1].set_title(f'Power Generation on a Cloudy Day ({cloudy_day})')

plt.tight_layout()
plt.show()

## 5. Summary of Weather Impact

Based on the analysis:
1. **Irradiation** is the dominant factor (Corr > 0.9). No irradiation = No power.
2. **Module Temperature** has a dual role: It increases with irradiation (good), but for a *fixed* amount of irradiation, higher temperatures slightly *reduce* efficiency.
3. **Ambient Temperature** acts as a baseline for module temperature.
4. **Plant Differences**: Plant 1 and Plant 2 have different power scales but follow the same physical patterns.